In [1]:
from flappy_agent import *
import numpy as np

C:\Users\arnar\AppData\Local\Programs\Python\Python313\Lib\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


In [2]:
# Fresh agent
agent = FlappyAgent([64, 64])
# Or load existing weights to continue training:
# agent = FlappyAgent(weights_path='flappy weights/flappy_ppo_trained.pt')

In [3]:
training_params = {
    'gamma': 0.999,
    'lam': 0.95,
    'clip_eps': 0.2,
    'value_coef': 0.001,
    'entropy_coef': 0.1,
    'max_grad_norm': np.inf,
    'learning_rate': 0.0001,
    'ppo_epochs': 5,
    'num_epochs': 5000,
    'games_per_epoch': 40,
    'minibatch_size': 128,
    'print_freq': 10
}

In [ ]:
print('Using parameters:')
for key, value in training_params.items():
    print(f'{key}: {value}')
agent.run_training(**training_params)

Epoch   170 | Avg Pipes:   17.47 | L_clip: -0.0205 | L_vf: 0.5823 | L_ent: 0.3628 | Loss: -0.0152


In [ ]:
torch.save(agent.network.state_dict(), 'flappy weights/flappy_ppo_trained.pt')
print("Weights saved to 'flappy weights/flappy_ppo_trained.pt'")

In [ ]:
import numpy as np

num_test_episodes = 100
scores = []

from ple.games.flappybird import FlappyBird
from ple import PLE
game = FlappyBird()
episode = PLE(game, fps=30, display_screen=False, force_fps=True)
episode.init()

for ep in range(num_test_episodes):
    agent.play_episode(episode, mode='Exploit')
    score = agent.memory[:, 9].sum().item()  # reward is at index 9 (8 state features + action + reward)
    scores.append(score)

scores = np.array(scores)
print(f"Average score: {scores.mean():.1f} +/- {scores.std():.1f}")
print(f"Max score: {scores.max():.1f}")
print(f"Min score: {scores.min():.1f}")

In [ ]:
import pygame
import pygame
import os

os.environ.pop('SDL_VIDEODRIVER', None)
pygame.quit()
pygame.init()

num_runs = 10
game_vis = FlappyBird()
p_vis = PLE(game_vis, fps=30, display_screen=True, force_fps=False)
p_vis.init()

for run in range(num_runs):
    p_vis.reset_game()
    total_reward = 0
    while not p_vis.game_over():
        state = state_to_tensor(p_vis.getGameState())
        action, _, _ = agent.get_action(state, mode='Exploit')
        reward = p_vis.act(p_vis.getActionSet()[action])
        total_reward += reward
    print(f"Run {run+1}/{num_runs} - Total reward: {total_reward:.1f}")

pygame.quit()
print("Done.")